# ASYA: 10 отдельных изображений вороны
Включи **Runtime → Change runtime type → T4 GPU**, затем **Runtime → Run all**. Бесплатный GPU зависит от доступности Colab. Каждая сцена генерируется отдельным вызовом модели. Используется SD-Turbo: качество ниже современных платных генераторов, но модель открытая. Выход: 10 PNG и ZIP.

In [ ]:
%pip -q install 'diffusers>=0.30,<0.38' 'transformers>=4.44,<5' accelerate safetensors


In [ ]:
# ASYA: 10 independent crow images, one file per prompt
import os, gc, zipfile, torch
from pathlib import Path
from PIL import Image
from diffusers import AutoPipelineForText2Image
from google.colab import files

if not torch.cuda.is_available():
    raise RuntimeError("GPU not available. Colab: Runtime → Change runtime type → T4 GPU. Free GPU availability is not guaranteed.")

output = Path("/content/crow_scenes")
output.mkdir(exist_ok=True)
model = "stabilityai/sd-turbo"
pipe = AutoPipelineForText2Image.from_pretrained(model, torch_dtype=torch.float16, variant="fp16")
pipe = pipe.to("cuda")
pipe.set_progress_bar_config(disable=False)

style = "Wildlife documentary photograph of a single anatomically accurate hooded crow (Corvus cornix), black head wings and tail, ash grey body, natural bird proportions, detailed feathers, cinematic low-key lighting, photographic realism, vertical portrait composition, no text, no watermark. "
prompts = [
"At dawn the crow stands on a rain soaked branch above a misty European forest, wide establishing shot.",
"Extreme close-up of the crow's dark eye and feather textures, wet glints, shallow depth of field.",
"The crow studies a pedestrian from a wet iron fence beside a quiet city street, human blurred behind.",
"Close shot of the crow tilting its head while observing the same hooded pedestrian from a safe distance.",
"The crow launches from a fence into flight, wings fully extended, dramatic overcast urban sky.",
"The crow discovers a whole walnut among fallen autumn leaves on wet cobblestones, ground-level shot.",
"The crow picks up the walnut carefully with its beak, visible claws and detailed pavement.",
"The crow drops the walnut onto a deserted road from a low height, dynamic action frame.",
"The crow waits beside a pedestrian crossing, walnut on road, distant vehicle out of focus, realistic city.",
"The crow eats pieces of a cracked walnut on the sidewalk at dusk, atmospheric final scene."
]
for i, scene in enumerate(prompts, 1):
    gen = torch.Generator(device="cuda").manual_seed(20261022 + i*97)
    image = pipe(prompt=style + scene, num_inference_steps=4, guidance_scale=0.0,
                 width=512, height=768, generator=gen).images[0]
    # Upscale for montage compatibility; this does not add native detail.
    image = image.resize((1080,1920), Image.Resampling.LANCZOS)
    target = output / f"scene_{i:02d}.png"
    image.save(target, optimize=True)
    print(f"{i}/10: {target}")
    del image
    torch.cuda.empty_cache()
archive = "/content/crow_10_independent_scenes.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(output.glob("*.png")):
        z.write(p, p.name)
print("Ready:", archive)
files.download(archive)



## Отправка изображений напрямую в GitHub (необязательно)
Запусти следующую ячейку после генерации. Создай **fine-grained personal access token** для репозитория `honeybadger-studio` с разрешением **Contents: Read and write**. Вставь токен в скрытый ввод; он не сохраняется в ноутбуке. Изображения попадут в `assets/`. Не публикуй токен и не сохраняй ноутбук с выводом, содержащим секреты.
**Автоматическая сборка запустится, когда в `assets/` также появится `voice.mp3`.**


In [ ]:
import base64, getpass, requests
from pathlib import Path
token = getpass.getpass("GitHub fine-grained token (Contents: read/write): ").strip()
if not token:
    raise ValueError("Token required")
repo = "todimka93-hub/honeybadger-studio"
session = requests.Session()
session.headers.update({"Authorization": f"Bearer {token}", "Accept":"application/vnd.github+json", "X-GitHub-Api-Version":"2022-11-28"})
try:
    for p in sorted(Path("/content/crow_scenes").glob("scene_*.png")):
        url = f"https://api.github.com/repos/{repo}/contents/assets/{p.name}"
        check = session.get(url, timeout=30)
        if check.status_code not in (200,404):
            raise RuntimeError(f"GitHub read {p.name}: {check.status_code} {check.text[:300]}")
        payload = {"message": f"Add generated crow scene {p.name}", "content": base64.b64encode(p.read_bytes()).decode()}
        if check.status_code == 200:
            payload["sha"] = check.json()["sha"]
        res = session.put(url, json=payload, timeout=120)
        if res.status_code not in (200,201):
            raise RuntimeError(f"GitHub upload {p.name}: {res.status_code} {res.text[:300]}")
        print("Uploaded:",p.name)
    print("10 images uploaded. If voice.mp3 exists in assets/, Build Shorts will start automatically.")
finally:
    session.headers.pop("Authorization",None)
    token = None

